In [22]:
import pandas as pd
import numpy as np
import sys

# import path and function.
sys.path.insert(0, '/home/zli333/hpchome/network_conditioning/codes/')
import network_functions as netf

# 1. Load main network data
net_original = pd.read_parquet('../hlm_data/structure/sangamon.gzip')

# 2. Calculate parent ('ds' column) - once
net_original['ds'] = -1 # Initialize 'ds' column
for k_index in net_original.index:
    num_parents = int(net_original.loc[k_index, 'np'])
    if num_parents > 0:
        parents_ids_values = [net_original.loc[k_index, f'us{j}'] for j in range(1, num_parents + 1)]
        for p_id in parents_ids_values:
            # Filter out invalid link IDs (e.g., 0 or -1) and ensure p_id is a valid index in net_original.
            if pd.notna(p_id) and p_id != 0 and p_id != -1 and p_id in net_original.index:
                net_original.loc[p_id, 'ds'] = k_index

# 3. Initialize results DataFrame
# This DataFrame will have LINKNO as index and a column for sub-watershed ID for each 'i'.
results_df = pd.DataFrame(index=net_original.index)
results_df.index.name = 'LINKNO'

# 4. Main loop for strmOrder thresholds [4, 5, 6, 7, 8]
for i_strm_order_threshold in range(4, 9):
    print(f"\nProcessing for strmOrder >= {i_strm_order_threshold}...")
    
    # Create a working copy for the current iteration (already contains 'ds' column).
    net_iter = net_original.copy() 

    # Filter by strmOrder to create 'n_filtered'.
    n_filtered = net_iter.loc[net_iter['strmOrder'] >= i_strm_order_threshold].copy()
    
    column_name_for_i = f'subw_{i_strm_order_threshold}' # New column name format

    if n_filtered.empty:
        print(f"No data for strmOrder >= {i_strm_order_threshold}. '{column_name_for_i}' will be NaN for this threshold.")
        results_df[column_name_for_i] = np.nan # Fill with NaN if no data to process.
        continue # Continue to the next i_strm_order_threshold.
        
    n_filtered = n_filtered.sort_values('DSContArea', ascending=False)

    # Calculate 'cum_up' on n_filtered.
    n_filtered.loc[:, 'cum_up'] = 1
    for idx_n in n_filtered.index:
        down_stream_reach_id = n_filtered.loc[idx_n, 'ds']
        if pd.notna(down_stream_reach_id) and down_stream_reach_id != -1 and down_stream_reach_id in n_filtered.index:
            n_filtered.loc[down_stream_reach_id, 'cum_up'] += 1
    
    # Sort 'net_iter' for sub-watershed processing .
    net_iter = net_iter.sort_values('DSContArea', ascending=False)
    
    sub_watershed_id_counter = 1
    # Initialize 'sub_watershed' column in net_iter for the current iteration.
    net_iter.loc[:, 'sub_watershed'] = sub_watershed_id_counter 

    # Determine LIDs for splitting from 'n_filtered' based on 'us1' and 'us2'.
    lids_for_splitting = []
    # Directly extract 'us1' and 'us2' values from rows meeting the criteria.
    potential_lids_df = n_filtered.loc[n_filtered['cum_up'] >= 3, ['us1', 'us2']]
    if not potential_lids_df.empty: # Check if any rows match before accessing .values.
        potential_lids = potential_lids_df.values.reshape(-1)
        valid_lids = [lid for lid in potential_lids if pd.notna(lid) and lid != 0 and lid != -1 and lid in net_iter.index]
        
        seen_lids = set()
        for lid in valid_lids:
            if lid not in seen_lids:
                lids_for_splitting.append(lid)
                seen_lids.add(lid)
    
    print(f"Identified {len(lids_for_splitting)} unique LIDs for sub-watershed splitting.")

    for lid_val in lids_for_splitting:
        sub_watershed_id_counter += 1
        _n_sub_df = netf.get_subwatershed(net_iter, lid_val) # Call external function.
        if _n_sub_df is not None and not _n_sub_df.empty:
            # Ensure all indices in _n_sub_df.index exist in net_iter.index.
            valid_sub_indices = _n_sub_df.index[_n_sub_df.index.isin(net_iter.index)]
            if not valid_sub_indices.empty:
                net_iter.loc[valid_sub_indices, 'sub_watershed'] = sub_watershed_id_counter
    
    # 4b. Store results for the current 'i' into results_df.
    # Assign the 'sub_watershed' Series from net_iter to results_df.
    results_df[column_name_for_i] = net_iter['sub_watershed']

# 5. Save the consolidated DataFrame to a single CSV file.
# results_df index is LINKNO. To make it a column in the CSV:
final_output_df = results_df.reset_index() 

output_filename = 'watershed_division_by_filtered_joints.csv'
watershed_division_by_filtered_joints
final_output_df.to_csv(output_filename, index=False)
print(f"\nConsolidated results saved to: {output_filename} (Shape: {final_output_df.shape})")

print("\nProcessing complete.")


Processing for strmOrder >= 4...
Identified 1384 unique LIDs for sub-watershed splitting.

Processing for strmOrder >= 5...
Identified 308 unique LIDs for sub-watershed splitting.

Processing for strmOrder >= 6...
Identified 70 unique LIDs for sub-watershed splitting.

Processing for strmOrder >= 7...
Identified 12 unique LIDs for sub-watershed splitting.

Processing for strmOrder >= 8...
Identified 2 unique LIDs for sub-watershed splitting.

Consolidated results saved to: linkno_subwatershed_ids_by_strmOrder_threshold.csv (Shape: (116779, 6))

Processing complete.
